# Data Preprocessing

In [1]:
import numpy as np
from sklearn.impute import KNNImputer
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from imblearn.over_sampling import SMOTENC

# --- 0. LOAD AND CLEAN DATA ---
df=pd.read_csv('Term_Project_Dataset_20K.csv')
df.loc[df['lecture_attendance_rate'] > 100, 'lecture_attendance_rate'] = np.nan
df.loc[df['stress_level'] < 0, 'stress_level'] = np.nan
df.loc[df['sleep_hours'] < 0, 'sleep_hours'] = np.nan

numerical_features = df.select_dtypes(include=np.number).columns.drop('final_score')
imputer = KNNImputer(n_neighbors=5)
df[numerical_features] = imputer.fit_transform(df[numerical_features])

cat_cols = ['gender', 'part_time_job', 'course_type']
df[cat_cols] = df[cat_cols].fillna('Unknown')
df = pd.get_dummies(df, columns=cat_cols,drop_first=True)


# --- 1. SETUP AND STRATIFIED SPLIT ---
target_cols = ['pass_fail', 'final_grade', 'final_score']

df_unlabeled = df[df[target_cols].isnull().any(axis=1)].copy()
df_known = df[df[target_cols].notnull().all(axis=1)].copy()
df_train_orig, df_test = train_test_split(
    df_known, 
    test_size=0.2, 
    random_state=42, 
    stratify=df_known['pass_fail']
)
print("Dataset Splits before pseudo labeling:")
print(f"Initial Training Set:    {len(df_train_orig)}")
print(f"Hold-out Test Set:       {len(df_test)}")
print(f"Unlabeled Set:           {len(df_unlabeled)}")
print(f"Total Dataset:           {len(df)}")


# --- 2. CLUSTERING FOR PSEUDO-LABELING ---
scaler_pre = StandardScaler()
X_train_pre = df_train_orig.drop(columns=target_cols)
X_unlabeled_pre = df_unlabeled.drop(columns=target_cols)

X_train_scaled_pre = scaler_pre.fit_transform(X_train_pre)
X_unlabeled_scaled_pre = scaler_pre.transform(X_unlabeled_pre)

k = 30 
kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
df_train_orig['cluster'] = kmeans.fit_predict(X_train_scaled_pre)
df_unlabeled['cluster'] = kmeans.predict(X_unlabeled_scaled_pre)

# --- 3. CALCULATE CLUSTER STATISTICS ---
cluster_stats = {}
for c_id in range(k):
    cluster_subset = df_train_orig[df_train_orig['cluster'] == c_id]
    if not cluster_subset.empty:
        mode_bin = cluster_subset['pass_fail'].mode()[0]
        cluster_stats[c_id] = {
            'binary': mode_bin,
            'multi': cluster_subset['final_grade'].mode()[0],
            'reg': cluster_subset['final_score'].mean(),
            'purity': (cluster_subset['pass_fail'] == mode_bin).mean()
        }

# --- 4. APPLY PSEUDO-LABELS ---
def apply_pseudo_labels(row):
    c_id = row['cluster']
    if c_id in cluster_stats and cluster_stats[c_id]['purity'] >= 0.95:
        return pd.Series([cluster_stats[c_id]['binary'], cluster_stats[c_id]['multi'], cluster_stats[c_id]['reg']])
    return pd.Series([np.nan, np.nan, np.nan])

df_unlabeled[target_cols] = df_unlabeled.apply(apply_pseudo_labels, axis=1)
df_pseudo_labeled = df_unlabeled.dropna(subset=target_cols).copy()

print("After Pseudo-Labeling:")
print(f"Pseudo-Labeled Set:      {len(df_pseudo_labeled)}")
print(f"Total Training Set:      {len(df_train_orig) + len(df_pseudo_labeled)}")
print(f"Total Dataset:           {len(df_train_orig) + len(df_pseudo_labeled) + len(df_test)}")
print(f"Pseudo-Labeling Added:   {len(df_pseudo_labeled)} samples")
# FINAL CONCATENATION
df_final_train = pd.concat([df_train_orig, df_pseudo_labeled], axis=0).reset_index(drop=True)
print(f"Final Training Set Size: {len(df_final_train)}")
# --- 5. FINAL FEATURE PREPARATION ---
features = [col for col in df_final_train.columns if col not in target_cols + ['cluster']]
scaler_mid = StandardScaler()
X_train_final_scaled = scaler_mid.fit_transform(df_final_train[features])
X_test_final_scaled = scaler_mid.transform(df_test[features])




Dataset Splits before pseudo labeling:
Initial Training Set:    14604
Hold-out Test Set:       3651
Unlabeled Set:           1745
Total Dataset:           20000
After Pseudo-Labeling:
Pseudo-Labeled Set:      612
Total Training Set:      15216
Total Dataset:           18867
Pseudo-Labeling Added:   612 samples
Final Training Set Size: 15216


In [2]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score

# --- 6. LASSO REGRESSION WITH CROSS-VALIDATION ---

# We use the final_score as our target for regression
y_train_reg = df_final_train['final_score']
y_test_reg = df_test['final_score']

# Initialize LassoCV
# cv=5: 5-fold cross-validation
# random_state=42: ensures reproducibility
lasso_cv = LassoCV(alphas=None, cv=5, max_iter=10000, random_state=42)

# Fit the model
lasso_cv.fit(X_train_final_scaled, y_train_reg)

# --- 7. EVALUATION ---

# Make predictions
y_pred = lasso_cv.predict(X_test_final_scaled)

# Calculate metrics
mse = mean_squared_error(y_test_reg, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test_reg, y_pred)

print("-" * 30)
print(f"LassoCV Results:")
print(f"Best Alpha (Regularization Strength): {lasso_cv.alpha_:.6f}")
print(f"R-squared Score: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")

# --- 8. FEATURE SELECTION ANALYSIS ---

# Map coefficients back to feature names
lasso_coeffs = pd.Series(lasso_cv.coef_, index=features)
selected_features = lasso_coeffs[lasso_coeffs != 0]
eliminated_features = lasso_coeffs[lasso_coeffs == 0]

print("-" * 30)
print(f"Total features: {len(features)}")
print(f"Features kept by Lasso: {len(selected_features)}")
print(f"Features eliminated: {len(eliminated_features)}")

# Display top 10 most influential features
print("\nTop 10 Influential Features:")
print(selected_features.abs().sort_values(ascending=False).head(10))

c:\Users\youss\Documents\ML Course Project\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1682: FutureWarning: 'alphas=None' is deprecated and will be removed in 1.9, at which point the default value will be set to 100. Set 'alphas=100' to silence this warning.
  warnings.warn(


------------------------------
LassoCV Results:
Best Alpha (Regularization Strength): 0.048194
R-squared Score: 0.7637
RMSE: 5.4071
------------------------------
Total features: 41
Features kept by Lasso: 20
Features eliminated: 21

Top 10 Influential Features:
midterm_score                 6.836688
quiz_avg_score                4.130588
lecture_attendance_rate       3.394336
assignment_submission_rate    3.389423
stress_level                  0.248604
part_time_job_Yes             0.053832
course_difficulty_rating      0.052436
sleep_hours                   0.039209
num_failed_courses            0.034445
teacher_experience_years      0.032718
dtype: float64


In [4]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

# --- 9. PREPARE TARGETS & ENCODING ---

# Fix for ValueError: Map 'Pass'/'Fail' strings to 1/0
binary_mapping = {'Pass': 1, 'Fail': 0}
y_train_bin = df_final_train['pass_fail'].map(binary_mapping).astype(int)
y_test_bin = df_test['pass_fail'].map(binary_mapping).astype(int)

# Multi-class target: Label encode grades (e.g., 'A', 'B' -> 0, 1)
le = LabelEncoder()
y_train_multi = le.fit_transform(df_final_train['final_grade'])
y_test_multi = le.transform(df_test['final_grade'])

# --- 10. BINARY CLASSIFICATION (pass_fail) ---
# Calculate ratio for scale_pos_weight
num_neg = (y_train_bin == 0).sum()
num_pos = (y_train_bin == 1).sum()
# Add a small epsilon to avoid division by zero if dataset is extremely small
scale_weight = num_neg / max(num_pos, 1) 

xgb_bin = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric='aucpr'
)

xgb_bin.fit(X_train_final_scaled, y_train_bin)
bin_preds = xgb_bin.predict(X_test_final_scaled)

# --- 11. MULTI-CLASS CLASSIFICATION (final_grade) ---
classes = np.unique(y_train_multi)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_multi)
weights_dict = dict(zip(classes, weights))
sample_weights = np.array([weights_dict[cls] for cls in y_train_multi])

xgb_multi = xgb.XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softprob',
    random_state=42
)

xgb_multi.fit(X_train_final_scaled, y_train_multi, sample_weight=sample_weights)
multi_preds = xgb_multi.predict(X_test_final_scaled)

# --- 12. PRINTING COMPREHENSIVE REPORTS ---

def print_model_report(y_true, y_pred, title, target_names=None):
    print(f"\n{'='*25} {title} {'='*25}")
    # Balanced accuracy is key for your imbalanced data
    print(f"Balanced Accuracy: {balanced_accuracy_score(y_true, y_pred):.4f}")
    print(f"Standard Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    # Text-based confusion matrix representation
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, y_pred))

# Print Results
print_model_report(y_test_bin, bin_preds, "BINARY: PASS/FAIL", target_names=['Fail', 'Pass'])
print_model_report(y_test_multi, multi_preds, "MULTI-CLASS: FINAL GRADE", target_names=le.classes_.astype(str))


========================= BINARY: PASS/FAIL =========================
Balanced Accuracy: 0.8527
Standard Accuracy: 0.9063

Classification Report:
              precision    recall  f1-score   support

        Fail       0.41      0.79      0.54       253
        Pass       0.98      0.91      0.95      3398

    accuracy                           0.91      3651
   macro avg       0.70      0.85      0.74      3651
weighted avg       0.94      0.91      0.92      3651

Confusion Matrix:
[[ 200   53]
 [ 289 3109]]

========================= MULTI-CLASS: FINAL GRADE =========================
Balanced Accuracy: 0.5056
Standard Accuracy: 0.4352

Classification Report:
              precision    recall  f1-score   support

           A       0.22      0.45      0.30        82
           B       0.22      0.48      0.30       291
           C       0.76      0.39      0.52      2572
           D       0.29      0.57      0.38       600
           F       0.20      0.63      0.31       106

 